In [1]:
import pandas as pd
from sqlalchemy import create_engine

### Step 1: Load the data from files

In [2]:
fournisseur_df = pd.read_csv('fournisseurs.csv', delimiter=";")
produits_df = pd.read_excel('Produit.xlsx')
cout_produit_df = pd.read_csv('cout_produits.csv', delimiter=";")
fournisseur_facture_df = pd.read_csv('Fournisseur_facture.csv', delimiter=";")

In [3]:
print("fournisseur_df columns:", fournisseur_df.columns)
fournisseur_df.head()

fournisseur_df columns: Index(['Code fournisseur', 'NomF', 'PrenomF', 'VilleF'], dtype='object')


,Code fournisseur,NomF,PrenomF,VilleF
0,P1111,Alami,Ali,Rabat
1,P2222,Msmoudi,Salah,Rabat
2,P3333,Imran,kamal,Kenitra
3,P4444,benchkroun,Mehdi,Oujda
4,P5555,laarbi,amine,Sale


In [4]:
print("produits_df columns:", produits_df.columns)
produits_df.head()

produits_df columns: Index(['Ref Produits', 'Nom', 'Categorie', 'quantite_achat', 'date_achat',
       'Code fournisseur'],
      dtype='object')


,Ref Produits,Nom,Categorie,quantite_achat,date_achat,Code fournisseur
0,C2700,Clavier,Lenovo,10,2001-01-10,P1111
1,C1000,Souris,Microsoft,26,2001-02-01,P2222
2,C1100,Ecran,Dell,16,2002-05-03,P3333
3,C1200,Ordinateur Portable,HP,29,2002-08-04,P4444
4,C1400,Ordinateur Portable,Acer,12,2002-10-09,P5555


In [5]:
print("cout_produit_df columns:", cout_produit_df.columns)
cout_produit_df.head()

cout_produit_df columns: Index(['idcout', 'Ref_produit', 'Montant_produit', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5'],
      dtype='object')


,idcout,Ref_produit,Montant_produit,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,1.0,C2700,100.0,NaN,NaN,NaN
1,2.0,C1000,40.0,NaN,NaN,NaN
2,3.0,C1100,230.0,NaN,NaN,NaN
3,4.0,C1200,540.0,NaN,NaN,NaN
4,5.0,C1400,700.0,NaN,NaN,NaN


In [6]:
print("fournisseur_facture_df columns:", fournisseur_facture_df.columns)
fournisseur_facture_df.head()

fournisseur_facture_df columns: Index(['facture_id', 'product_id', 'fournisseur_id', 'Qt_Produit'], dtype='object')


,facture_id,product_id,fournisseur_id,Qt_Produit
0,1,C2700,P1111,10
1,2,C1000,P2222,26
2,3,C1100,P3333,16
3,4,C1200,P4444,29
4,5,C1400,P5555,12


### Step 2: Transform the data

In [7]:
# clean cout_produits dataset :
cout_produit_df = cout_produit_df.drop(columns=["Unnamed: 3", "Unnamed: 4", "Unnamed: 5"])
cout_produit_df = cout_produit_df.dropna()

# Create fournisseur dimension
fournisseur_df['nom_prenom_fournisseur'] = fournisseur_df['NomF'] + ' ' + fournisseur_df['PrenomF']
fournisseur_dim = fournisseur_df[['Code fournisseur', 'nom_prenom_fournisseur', 'VilleF']].rename(columns={'Code fournisseur': 'fournisseur_id', 'VilleF': 'ville_fournisseur'})

# Create product dimension
produits_df = produits_df.rename(columns={'Ref Produits': 'produit_id', 'Nom': 'nom_produit', 'Categorie': 'categorie_produit'})
product_dim = produits_df[['produit_id', 'nom_produit', 'categorie_produit', 'quantite_achat', 'Code fournisseur']].rename(columns={'Code fournisseur': 'code_fournisseur'})

# Create time dimension
produits_df['date_achat'] = pd.to_datetime(produits_df['date_achat'])
time_dim = produits_df[['date_achat']].drop_duplicates()
time_dim['jour'] = time_dim['date_achat'].dt.day
time_dim['mois'] = time_dim['date_achat'].dt.month
time_dim['annee'] = time_dim['date_achat'].dt.year
time_dim['temps_id'] = range(1, len(time_dim) + 1)
time_dim['date_achat'] = produits_df['date_achat']
# time_dim = time_dim[['temps_id', 'jour', 'mois', 'annee']]


In [8]:
# Create fact table
fact_df = fournisseur_facture_df.merge(produits_df[['produit_id', 'date_achat']], left_on='product_id', right_on='produit_id')
fact_df = fact_df.merge(cout_produit_df[['Ref_produit', 'Montant_produit']], left_on='product_id', right_on='Ref_produit')

# Merge fact_df with time_dim on 'date_achat'
fact_df = fact_df.merge(time_dim, on='date_achat')

fact_df['nombre_factures'] = 1
fact_df['cout_produit'] = fact_df['Montant_produit'] * fact_df['Qt_Produit']

fact_table = fact_df[['facture_id', 'fournisseur_id', 'product_id', 'temps_id', 'nombre_factures', 'cout_produit']]
fact_table = fact_table.rename(columns={'product_id': 'produit_id'})

# -----
time_dim = time_dim[['temps_id', 'jour', 'mois', 'annee']]

In [9]:
fact_table.head()

,facture_id,fournisseur_id,produit_id,temps_id,nombre_factures,cout_produit
0,1,P1111,C2700,1,1,1000.0
1,2,P2222,C1000,2,1,1040.0
2,3,P3333,C1100,3,1,3680.0
3,4,P4444,C1200,4,1,15660.0
4,5,P5555,C1400,5,1,8400.0


In [10]:
time_dim.head()

,temps_id,jour,mois,annee
0,1,10,1,2001
1,2,1,2,2001
2,3,3,5,2002
3,4,4,8,2002
4,5,9,10,2002


In [11]:
fournisseur_dim.head()

,fournisseur_id,nom_prenom_fournisseur,ville_fournisseur
0,P1111,Alami Ali,Rabat
1,P2222,Msmoudi Salah,Rabat
2,P3333,Imran kamal,Kenitra
3,P4444,benchkroun Mehdi,Oujda
4,P5555,laarbi amine,Sale


In [12]:
product_dim.head()

,produit_id,nom_produit,categorie_produit,quantite_achat,code_fournisseur
0,C2700,Clavier,Lenovo,10,P1111
1,C1000,Souris,Microsoft,26,P2222
2,C1100,Ecran,Dell,16,P3333
3,C1200,Ordinateur Portable,HP,29,P4444
4,C1400,Ordinateur Portable,Acer,12,P5555


### Step 3: Load the transformed data into MySQL

In [14]:
# Database connection
engine = create_engine('mysql+pymysql://root:@localhost:3306/bi_data_warehouse')

# Write DataFrames to MySQL
fournisseur_dim.to_sql(name='dim_fournisseur', con=engine, if_exists='replace', index=False)
product_dim.to_sql(name='dim_produit', con=engine, if_exists='replace', index=False)
time_dim.to_sql(name='dim_temps', con=engine, if_exists='replace', index=False)
fact_table.to_sql(name='fact_ventes', con=engine, if_exists='replace', index=False)

print("Data successfully loaded into the database")


Data successfully loaded into the database
